In [1]:
import os
import requests
import pandas as pd

from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("FANTASYPROS_API_KEY")

Check API key loaded. DO NOT print it.

In [2]:
assert API_KEY is not None, "FantasyPros API key not found"
print("FantasyPros API key loaded successfully.")

FantasyPros API key loaded successfully.


Assemple API request components for player data

In [3]:
BASE_URL = "https://api.fantasypros.com/public/v2/json"

url = f"{BASE_URL}/nfl/2026/consensus-rankings"

headers = {
    "x-api-key": API_KEY
}

params = {
    "position": "ALL",
    "type": "ADP",
    "scoring": "HALF"
}

# Inspect, but NOT the API Key
print("URL:", url)
print("Parameters:", params)
print("API key configured:", API_KEY is not None)

URL: https://api.fantasypros.com/public/v2/json/nfl/2026/consensus-rankings
Parameters: {'position': 'ALL', 'type': 'ADP', 'scoring': 'HALF'}
API key configured: True


Make a test API call.  

NOTE - ONLY RUN THIS ONCE! Only 50 API calls/day. 

In [4]:
response = requests.get(
    url,
    headers=headers,
    params=params,
    timeout=30
)

# Inspect the player data response

In [5]:
data = response.json()

print("Status:", response.status_code)
print("Tier:", data.get("tier"))
print("Public API limited:", data.get("public_api_limited"))
print("Count:", data.get("count"))
print("Limit:", data.get("limit"))
print("Players returned:", len(data.get("players", [])))

Status: 200
Tier: premium
Public API limited: True
Count: 340
Limit: None
Players returned: 340


In [12]:
for key, value in data.items():
    if key != "players":
        print(f"{key}: {value}")

sport: NFL
type: ADP Half PPR
ranking_type_name: adp
year: 2026
week: 0
position_id: ALL
scoring: HALF
filters: 236,439,4350
count: 340
total_experts: 3
last_updated: 8/27
last_updated_ts: 1787858414
public_api_limited: True
tier: premium


In [ ]:
# # Full response data
# data

## Save player data locally

In [7]:
import json
from pathlib import Path

raw_path = Path("../data/raw/fantasypros_adp_2026_half.json")

with open(raw_path, "w") as f:
    json.dump(data, f, indent=2)

print(f"Saved raw response to: {raw_path}")

Saved raw response to: ../data/raw/fantasypros_adp_2026_half.json


Now confirm it exists & saved

In [8]:
print(raw_path.exists())
print(f"{raw_path.stat().st_size / 1024:.1f} KB")

True
291.5 KB


# Inspect Player Data

In [9]:
print(len(data["players"]))
data["players"][0]

340


{'player_id': 22968,
 'player_name': 'Jahmyr Gibbs',
 'sportsdata_id': 'fef9457e-6497-47de-9bf2-cc3b95929375',
 'player_team_id': 'DET',
 'player_position_id': 'RB',
 'player_positions': 'RB',
 'player_short_name': 'J. Gibbs',
 'player_eligibility': 'RB',
 'player_yahoo_positions': 'RB',
 'player_page_url': 'https://www.fantasypros.com/nfl/players/jahmyr-gibbs.php',
 'player_filename': 'jahmyr-gibbs.php',
 'player_yahoo_id': '40059',
 'cbs_player_id': '3162723',
 'player_bye_week': '6',
 'player_owned_avg': 99.5,
 'player_owned_espn': 99.9,
 'player_owned_yahoo': 100,
 'player_ecr_delta': None,
 'rank_ecr': 1,
 'rank_min': '1',
 'rank_max': '1',
 'rank_ave': '1.00',
 'rank_std': '0.00',
 'pos_rank': 'RB1',
 'tier': 1}

# Inspect ADP source metadata

In [13]:
experts_url = f"{BASE_URL}/nfl/2026/rankings/experts"

experts_params = {
    "position": "ALL",
    "type": "ADP",
    "scoring": "HALF"
}

print("URL:", experts_url)
print("Parameters:", experts_params)

URL: https://api.fantasypros.com/public/v2/json/nfl/2026/rankings/experts
Parameters: {'position': 'ALL', 'type': 'ADP', 'scoring': 'HALF'}


API CALL! - ONLY RUN ONCE!   
This is for platform-specific ADP data. 

In [14]:
experts_response = requests.get(
    experts_url,
    headers=headers,
    params=experts_params,
    timeout=30
)

print("Status code:", experts_response.status_code)

Status code: 200


In [15]:
experts_data = experts_response.json()

print(type(experts_data))
print(experts_data.keys())

<class 'dict'>
dict_keys(['sport', 'count', 'season', 'week', 'accuracy_weekly_season', 'accuracy_draft_season', 'accuracy_weekly_last_season', 'experts', 'public_api_limited', 'tier'])


In [16]:
print("Expert count:", len(experts_data["experts"]))

Expert count: 0


In [17]:
experts_data["experts"]

[]

## Retry Player Data with "Experts" 
ADP Metadata did not return "expert" data across platforms. 

In [27]:
params_with_experts = {
    "position": "ALL",
    "type": "ADP",
    "scoring": "HALF",
    "experts": "show"
}

API CALL BELOW! - Only run this once

In [28]:
response_with_experts = requests.get(
    url,                    # original consensus-rankings URL
    headers=headers,
    params=params_with_experts,
    timeout=30
)

In [20]:
data_with_experts = response_with_experts.json()

print("Status:", response_with_experts.status_code)
print("Filters:", data_with_experts.get("filters"))
print("Total experts:", data_with_experts.get("total_experts"))
print("Expert names:", data_with_experts.get("expert_name"))
print("Experts available:", data_with_experts.get("experts_available"))

Status: 200
Filters: 236,439,4350
Total experts: 3
Expert names: None
Experts available: None


In [22]:
print(data_with_experts.keys())

for key in [
    "expert_name",
    "expert_pub",
    "expert_twitter",
    "experts_available"
]:
    print(key, "->", key in data_with_experts, data_with_experts.get(key))

dict_keys(['sport', 'type', 'ranking_type_name', 'year', 'week', 'position_id', 'scoring', 'filters', 'count', 'total_experts', 'last_updated', 'players', 'last_updated_ts', 'expert_pub', 'expert_names', 'expert_twitter', 'public_api_limited', 'tier'])
expert_name -> False None
expert_pub -> True {'236': '2026-08-27 19:20:05', '439': '2026-08-27 05:30:06', '4350': '2026-08-27 19:20:14'}
expert_twitter -> True {'236': None, '439': None, '4350': 'SleeperHQ'}
experts_available -> False None


## Explore source IDs by "Expert" (platform)
Expert 4350 == Sleeper.  

3 API Calls below!

In [23]:
source_ids = [236, 439, 4350]

source_responses = {}

for source_id in source_ids:
    source_params = {
        "position": "ALL",
        "type": "ADP",
        "scoring": "HALF",
        "filters": str(source_id)
    }

    r = requests.get(
        url,
        headers=headers,
        params=source_params,
        timeout=30
    )

    print(source_id, r.status_code)

    source_responses[source_id] = r.json()

236 200
439 200
4350 200


Inspect responses

In [24]:
for source_id, source_data in source_responses.items():
    print(
        source_id,
        "count:", source_data.get("count"),
        "filters:", source_data.get("filters"),
        "total_experts:", source_data.get("total_experts")
    )

236 count: 340 filters: 236,439,4350 total_experts: 3
439 count: 340 filters: 236,439,4350 total_experts: 3
4350 count: 340 filters: 236,439,4350 total_experts: 3


## Still not what we want --> try "experts":"available"

In [29]:
params_available = {
    "position": "ALL",
    "type": "ADP",
    "scoring": "HALF",
    "experts": "available"
}

In [30]:
response_available = requests.get(
    url,
    headers=headers,
    params=params_available,
    timeout=30
)

In [31]:
available_data = response_available.json()

print("Status:", response_available.status_code)
print("Total experts:", available_data.get("total_experts"))
print("Filters:", available_data.get("filters"))
print("Experts available type:", type(available_data.get("experts_available")))
print("Experts available:", available_data.get("experts_available"))

Status: 200
Total experts: 3
Filters: 236,439,4350
Experts available type: <class 'dict'>
Experts available: {'total': 3, 'included': [236, 439, 4350], 'excluded': [], 'last_update': 1787858414}


# Now try "ranknings" endpoint

In [32]:
rankings_url = f"{BASE_URL}/nfl/2026/rankings"

rankings_params = {
    "week": 0
}

print("URL:", rankings_url)
print("Parameters:", rankings_params)

URL: https://api.fantasypros.com/public/v2/json/nfl/2026/rankings
Parameters: {'week': 0}


API Call!

In [33]:
rankings_response = requests.get(
    rankings_url,
    headers=headers,
    params=rankings_params,
    timeout=30
)

print("Status:", rankings_response.status_code)

Status: 200


### Inspect rankings response

In [34]:
rankings_data = rankings_response.json()

print(type(rankings_data))
print(rankings_data.keys())

<class 'dict'>
dict_keys(['sport', 'count', 'season', 'week', 'experts', 'players', 'ecr_experts', 'public_api_limited', 'tier'])


In [35]:
print("experts type:", type(rankings_data["experts"]))
print("players type:", type(rankings_data["players"]))
print("ecr_experts type:", type(rankings_data["ecr_experts"]))

experts type: <class 'dict'>
players type: <class 'list'>
ecr_experts type: <class 'dict'>


In [36]:
print("experts count:", len(rankings_data["experts"]))
print("players count:", len(rankings_data["players"]))

experts count: 8
players count: 1691


In [38]:
rankings_data["experts"].keys()

dict_keys(['WK1-STD', 'WK1-PPR', 'WK1-HALF', 'STD', 'PPR', 'HALF', 'DYN', 'BB-HALF'])

In [39]:
rankings_data["players"][0]

{'id': 8000,
 'player_name': 'Arizona Cardinals',
 'short_name': 'A. Cardinals',
 'first_name': 'Arizona',
 'last_name': 'Cardinals',
 'reverse_name': 'Cardinals, Arizona',
 'position_id': 'DST',
 'positions': ['DST'],
 'team_id': 'ARI',
 'filename': 'http://www.fantasypros.com/nfl/players/arizona-defense.php',
 'rank': {'ECR': {'WK1-STD': {'DST': 32},
   'WK1-PPR': {'DST': 32},
   'WK1-HALF': {'DST': 32},
   'STD': {'DST': 32},
   'PPR': {'DST': 32, 'ALL': 456},
   'HALF': {'DST': 32, 'ALL': 441},
   'DYN': {'ALL': 422, 'DST': 28},
   'BB-HALF': {'DST': 29}},
  'ADP': {'BB-HALF': {'ALL': 515, 'DST': 32}}}}

In [40]:
print(rankings_data["ecr_experts"].keys())

dict_keys(['WK1-STD', 'WK1-PPR', 'WK1-HALF', 'STD', 'PPR', 'HALF', 'DYN', 'BB-HALF'])


In [41]:
print(type(rankings_data["experts"]["HALF"]))
rankings_data["experts"]["HALF"]

<class 'dict'>


{'ALL': 109, 'RB': 107, 'WR': 109, 'TE': 104, 'OP': 104}

In [42]:
print(type(rankings_data["ecr_experts"]["HALF"]))
rankings_data["ecr_experts"]["HALF"]

<class 'dict'>


{'ALL': [4307,
  4317,
  7565,
  5446,
  96,
  7639,
  2716,
  1204,
  22,
  1139,
  692,
  678,
  2598,
  7604,
  7666,
  1408,
  747,
  317,
  9,
  1080,
  4338,
  960,
  3688,
  1402,
  2373,
  285,
  7,
  6,
  2658,
  552,
  2340,
  4439,
  3585,
  284,
  381,
  690,
  101,
  279,
  1576,
  3354,
  4616,
  7254,
  1152,
  4176,
  65,
  3379,
  1661,
  3514,
  4164,
  5001,
  7630,
  2932,
  670,
  3112,
  7648,
  394,
  3468,
  23,
  2780,
  1508,
  636,
  2694,
  2639,
  7590,
  684,
  4404,
  5874,
  453,
  7542,
  4408,
  3666,
  763,
  4160,
  2747,
  2559,
  1667,
  3612,
  2743,
  647,
  7662,
  1046,
  5598,
  5662,
  7555,
  3950,
  7616,
  562,
  873,
  3747,
  7671,
  5546,
  2791,
  2751,
  58,
  7266,
  3556,
  908,
  1138,
  3467,
  4355,
  487,
  7688,
  3404,
  1549,
  4179,
  3326,
  7546,
  5626,
  2475],
 'RB': [4307,
  4317,
  7565,
  5446,
  1204,
  96,
  7639,
  2716,
  22,
  1139,
  692,
  678,
  2598,
  7604,
  7666,
  1408,
  9,
  747,
  317,
  1080,
  4338,

In [43]:
gibbs = next(
    player for player in rankings_data["players"]
    if player["player_name"] == "Jahmyr Gibbs"
)

gibbs["rank"]

{'ECR': {'WK1-STD': {'RB': 1, 'FLX': 1, 'OP': 11},
  'WK1-PPR': {'RB': 1, 'FLX': 1, 'OP': 1},
  'WK1-HALF': {'RB': 1, 'FLX': 1, 'OP': 3},
  'STD': {'ALL': 1, 'RB': 1, 'OP': 7},
  'PPR': {'ALL': 2, 'RB': 1, 'OP': 8},
  'HALF': {'ALL': 1, 'RB': 1, 'OP': 7},
  'DYN': {'ALL': 5, 'RB': 2, 'OP': 11},
  'BB-HALF': {'ALL': 1, 'RB': 1, 'FLX': 1}},
 'ADP': {'BB-HALF': {'ALL': 1, 'RB': 1, 'OP': 1}}}